
# Earnings Call Delta Tracker — Infosys MVP

**Goal:** Compare 3 Infosys earnings-call PDFs and identify what changed between quarters.

### Input PDFs
- `INFY_Q3FY25.pdf`
- `INFY_Q4FY25.pdf`
- `INFY_Q1FY26.pdf`

### MVP outputs
1. Extract and clean transcript text
2. Split Management Commentary vs Q&A where possible
3. Extract important sentences
4. Compare latest call against previous calls using sentence embeddings
5. Detect new / dropped / materially changed themes
6. Measure financial tone using FinBERT
7. Produce a compact analyst-style "What Changed?" report

**Runtime:** Google Colab + NVIDIA T4 GPU

> This notebook intentionally avoids fragile dependency combinations. It pins the main ML stack and uses CPU-safe fallbacks where needed.


In [3]:

# ============================================================
# 1. INSTALL — PINNED COLAB-FRIENDLY STACK
# ============================================================

!pip -q install \
    "torch==2.6.0" \
    "transformers==4.49.0" \
    "sentence-transformers==3.4.1" \
    "pymupdf==1.25.3" \
    "pandas==2.2.3" \
    "numpy==1.26.4" \
    "scikit-learn==1.6.1" \
    "accelerate==1.4.0" \
    "safetensors==0.5.3" \
    "nltk==3.9.1" \
    "tabulate==0.9.0" \
    "matplotlib==3.10.0"

print("Installation complete.")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... canceled
ERROR: Operation cancelled by user
Installation complete.


In [8]:

# ============================================================
# 2. IMPORTS + ENVIRONMENT CHECK
# ============================================================
!pip install PyMuPDF
import os
import re
import gc
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import fitz

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("WARNING: GPU not detected. Runtime will still work, but slower.")

BASE_DIR = Path("/content/earnings_delta")
PDF_DIR = BASE_DIR / "pdfs"
OUTPUT_DIR = BASE_DIR / "outputs"
CACHE_DIR = BASE_DIR / "cache"

for d in [PDF_DIR, OUTPUT_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Working directory:", BASE_DIR)


PyTorch: 2.11.0+cu128
CUDA available: True
Device: cuda
GPU: Tesla T4
VRAM: 14.56 GB
Working directory: /content/earnings_delta



## 3. Upload the three PDFs

Run the next cell and upload your files.

Recommended filenames:

```text
INFY_Q3FY25.pdf
INFY_Q4FY25.pdf
INFY_Q1FY26.pdf
```


In [9]:

# ============================================================
# 3. UPLOAD PDFs
# ============================================================

from google.colab import files

uploaded = files.upload()

for filename, data in uploaded.items():
    target = PDF_DIR / filename
    with open(target, "wb") as f:
        f.write(data)

print("\nUploaded files:")
for p in sorted(PDF_DIR.glob("*.pdf")):
    print(" -", p.name)


Saving INFY_Q1FY26.pdf to INFY_Q1FY26.pdf
Saving INFY_Q3FY25.pdf to INFY_Q3FY25.pdf
Saving INFY_Q4FY25.pdf to INFY_Q4FY25.pdf

Uploaded files:
 - INFY_Q1FY26.pdf
 - INFY_Q3FY25.pdf
 - INFY_Q4FY25.pdf


In [10]:

# ============================================================
# 4. PDF TEXT EXTRACTION
# ============================================================

def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")
        text = re.sub(r"\s+", " ", text).strip()

        if text:
            pages.append({
                "page": page_num,
                "text": text
            })

    doc.close()
    return pd.DataFrame(pages)


pdf_tables = {}

for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    df = extract_pdf_text(pdf_path)
    pdf_tables[pdf_path.name] = df

    print(f"{pdf_path.name}: {len(df)} pages extracted")
    print("Characters:", df["text"].str.len().sum())
    print("-" * 60)


INFY_Q1FY26.pdf: 25 pages extracted
Characters: 50546
------------------------------------------------------------
INFY_Q3FY25.pdf: 25 pages extracted
Characters: 47827
------------------------------------------------------------
INFY_Q4FY25.pdf: 27 pages extracted
Characters: 50686
------------------------------------------------------------


In [11]:

# ============================================================
# 5. NORMALIZE TRANSCRIPT
# ============================================================

def normalize_text(text):
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\bPage\s+\d+\b", " ", text, flags=re.I)
    return text.strip()


def build_document(pdf_df):
    rows = []

    for _, row in pdf_df.iterrows():
        clean = normalize_text(row["text"])

        if len(clean) >= 30:
            rows.append({
                "page": int(row["page"]),
                "text": clean
            })

    return pd.DataFrame(rows)


documents = {}

for filename, df in pdf_tables.items():
    documents[filename] = build_document(df)

for filename, df in documents.items():
    print(filename, "->", len(df), "usable pages")


INFY_Q1FY26.pdf -> 25 usable pages
INFY_Q3FY25.pdf -> 25 usable pages
INFY_Q4FY25.pdf -> 27 usable pages



## 6. Convert transcript into sentences

For the MVP, we deliberately use a simple sentence splitter instead of adding another heavyweight NLP dependency.

Later we can replace this with a finance-specific sentence segmentation model.


In [12]:

# ============================================================
# 6. SENTENCE SEGMENTATION
# ============================================================

def split_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()

    # Conservative sentence splitting
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9"“])', text)

    cleaned = []

    for s in parts:
        s = s.strip()

        if len(s) < 30:
            continue

        # Remove obvious transcript noise
        if re.fullmatch(r"\d+", s):
            continue

        cleaned.append(s)

    return cleaned


def document_to_sentences(page_df, filename):
    rows = []

    for _, row in page_df.iterrows():
        for sentence in split_sentences(row["text"]):
            rows.append({
                "file": filename,
                "page": row["page"],
                "sentence": sentence
            })

    return pd.DataFrame(rows)


sentence_tables = {}

for filename, df in documents.items():
    sentence_tables[filename] = document_to_sentences(df, filename)
    print(filename, "->", len(sentence_tables[filename]), "sentences")


INFY_Q1FY26.pdf -> 476 sentences
INFY_Q3FY25.pdf -> 482 sentences
INFY_Q4FY25.pdf -> 453 sentences


In [13]:

# ============================================================
# 7. REMOVE COMMON BOILERPLATE / IRRELEVANT TEXT
# ============================================================

BOILERPLATE_PATTERNS = [
    r"safe harbor",
    r"forward[- ]looking statements",
    r"thank you for joining",
    r"operator",
    r"good morning everyone",
    r"good evening everyone",
    r"welcome to",
    r"we will now begin",
]

def is_boilerplate(sentence):
    s = sentence.lower()

    for pattern in BOILERPLATE_PATTERNS:
        if re.search(pattern, s):
            return True

    return False


for filename, df in sentence_tables.items():
    df["is_boilerplate"] = df["sentence"].apply(is_boilerplate)
    sentence_tables[filename] = df[~df["is_boilerplate"]].copy()

    print(filename, "remaining:", len(sentence_tables[filename]))


INFY_Q1FY26.pdf remaining: 470
INFY_Q3FY25.pdf remaining: 476
INFY_Q4FY25.pdf remaining: 447



# 8. Load Sentence Transformer

We use `all-MiniLM-L6-v2` for the MVP.

Why?

- Fast on a T4
- Small
- Good enough for semantic matching
- Easy to replace later with a stronger finance-specific embedding model

The purpose here is not to generate a summary. It is to answer:

> **Which statement in the latest call is semantically different from the previous quarter?**


In [14]:

# ============================================================
# 8. SENTENCE EMBEDDINGS
# ============================================================

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=DEVICE
)

print("Embedding model loaded on:", DEVICE)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded on: cuda


In [15]:

# ============================================================
# 9. EMBED EACH CALL
# ============================================================

for filename, df in sentence_tables.items():

    cache_file = CACHE_DIR / f"{Path(filename).stem}_embeddings.npy"

    sentences = df["sentence"].tolist()

    if cache_file.exists():
        embeddings = np.load(cache_file)
        print("Loaded cache:", filename)
    else:
        embeddings = embedder.encode(
            sentences,
            batch_size=64,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        np.save(cache_file, embeddings)

    df["embedding_index"] = range(len(df))
    df.attrs["embeddings"] = embeddings

    sentence_tables[filename] = df

    print(filename, embeddings.shape)


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

INFY_Q1FY26.pdf (470, 384)


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

INFY_Q3FY25.pdf (476, 384)


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

INFY_Q4FY25.pdf (447, 384)


In [17]:

# ============================================================
# 10. QUARTER ORDER
# ============================================================

# Explicit ordering avoids relying on filenames.

QUARTER_ORDER = {
    "INFY_Q3FY25.pdf": 1,
    "INFY_Q4FY25.pdf": 2,
    "INFY_Q1FY26.pdf": 3,
}

available = list(sentence_tables.keys())

missing = [x for x in QUARTER_ORDER if x not in available]

if missing:
    print("These expected filenames were not found:")
    for x in missing:
        print(" -", x)
    print("\nIf your filenames differ, edit QUARTER_ORDER above.")

ordered_files = [
    x for x in sorted(available, key=lambda x: QUARTER_ORDER.get(x, 999))
]

print("Comparison order:")
for x in ordered_files:
    print(" →", x)


Comparison order:
 → INFY_Q3FY25.pdf
 → INFY_Q4FY25.pdf
 → INFY_Q1FY26.pdf



# 11. Semantic Delta Detection

For every sentence in the latest call:

1. Find the most semantically similar sentence from the previous call.
2. Calculate similarity.
3. Low similarity means the statement is potentially **new or materially changed**.
4. High similarity means the topic was probably repeated.

This is the first core component of the Delta Tracker.


In [18]:

# ============================================================
# 11. COMPARE TWO CALLS
# ============================================================

def compare_calls(old_file, new_file, similarity_threshold=0.72):

    old_df = sentence_tables[old_file]
    new_df = sentence_tables[new_file]

    old_embeddings = old_df.attrs["embeddings"]
    new_embeddings = new_df.attrs["embeddings"]

    similarity_matrix = np.matmul(new_embeddings, old_embeddings.T)

    best_indices = similarity_matrix.argmax(axis=1)
    best_scores = similarity_matrix.max(axis=1)

    result = new_df.copy()

    result["matched_old_index"] = best_indices
    result["similarity"] = best_scores

    result["matched_old_sentence"] = [
        old_df.iloc[idx]["sentence"]
        for idx in best_indices
    ]

    result["matched_old_page"] = [
        old_df.iloc[idx]["page"]
        for idx in best_indices
    ]

    result["delta_score"] = 1 - result["similarity"]

    result["change_type"] = np.where(
        result["similarity"] < similarity_threshold,
        "POTENTIAL_NEW_OR_CHANGED",
        "SIMILAR"
    )

    return result.sort_values(
        "delta_score",
        ascending=False
    ).reset_index(drop=True)


latest_file = ordered_files[-1]
previous_file = ordered_files[-2]

delta_q1 = compare_calls(
    previous_file,
    latest_file
)

print("Comparing:")
print(previous_file, "→", latest_file)

display(
    delta_q1[
        [
            "page",
            "similarity",
            "delta_score",
            "change_type",
            "sentence",
            "matched_old_sentence"
        ]
    ].head(20)
)


Comparing:
INFY_Q4FY25.pdf → INFY_Q1FY26.pdf


,page,similarity,delta_score,change_type,sentence,matched_old_sentence
0,9,0.237453,0.762547,POTENTIAL_NEW_OR_CHANGED,I do not think it is making it easier or more ...,"So at this point-in-time, the endeavour is to ..."
1,13,0.323006,0.676994,POTENTIAL_NEW_OR_CHANGED,I wanted to ask on geographies.,"Region-wise, we signed 12 large deals each in ..."
2,22,0.330415,0.669585,POTENTIAL_NEW_OR_CHANGED,"Or are there any, say, one-off events which po...","Having said that, on the ramped downs, we have..."
3,10,0.335268,0.664732,POTENTIAL_NEW_OR_CHANGED,So that factor does not change.,"So as a composite, what we saw last year, 4.2%..."
4,13,0.339736,0.660264,POTENTIAL_NEW_OR_CHANGED,Can you comment on North America?,"Region-wise, we signed 12 large deals each in ..."
5,21,0.345300,0.654701,POTENTIAL_NEW_OR_CHANGED,Is there any such trend across geographies and...,We have had ramp-ups of many of the mega deals...
6,16,0.353366,0.646634,POTENTIAL_NEW_OR_CHANGED,Like how did you kind of arrive at that conclu...,So there were uncertainties around that.
7,22,0.366505,0.633495,POTENTIAL_NEW_OR_CHANGED,So I think all of that is reflected in SG&A.,So it is an existing JV where we have just ind...
8,22,0.369401,0.630599,POTENTIAL_NEW_OR_CHANGED,"On the SG&A, it is multiple factors.",All of these are areas of interest and strateg...
9,14,0.370233,0.629767,POTENTIAL_NEW_OR_CHANGED,"Bryan Bergin Okay, understood.",Next question is from the line of Jonathan Lee...


In [19]:

# ============================================================
# 12. FINANCIAL KEYWORD FILTER
# ============================================================

FINANCE_KEYWORDS = [
    "revenue", "growth", "margin", "ebit", "ebitda",
    "profit", "guidance", "outlook", "demand",
    "deal", "order", "bookings", "pipeline",
    "capex", "cash flow", "attrition", "utilization",
    "pricing", "cost", "headwind", "tailwind",
    "recovery", "client", "customers", "volume",
    "currency", "wage", "hiring", "genai", "ai",
    "artificial intelligence", "uncertainty"
]

def financial_relevance(sentence):
    s = sentence.lower()

    hits = [
        keyword for keyword in FINANCE_KEYWORDS
        if keyword in s
    ]

    return hits


delta_q1["keyword_hits"] = delta_q1["sentence"].apply(financial_relevance)
delta_q1["keyword_count"] = delta_q1["keyword_hits"].str.len()

important_changes = delta_q1[
    (delta_q1["change_type"] == "POTENTIAL_NEW_OR_CHANGED") &
    (delta_q1["keyword_count"] > 0)
].copy()

important_changes["priority_score"] = (
    important_changes["delta_score"] *
    (1 + np.log1p(important_changes["keyword_count"]))
)

important_changes = important_changes.sort_values(
    "priority_score",
    ascending=False
)

display(
    important_changes[
        [
            "page",
            "priority_score",
            "keyword_hits",
            "sentence",
            "matched_old_sentence"
        ]
    ].head(20)
)


,page,priority_score,keyword_hits,sentence,matched_old_sentence
30,6,1.218667,"[growth, deal]","We are partnering with GCCs, both in setup and...",Are you having any conversations with clients ...
136,22,1.218316,"[growth, margin, hiring, ai]",The hiring for the S&M mainly to improve our g...,"In terms of your margin walk, you have said th..."
102,18,1.175939,"[margin, ebit, ai]","And the last question is, I think in the press...","Conclusion from my side; first, we are delight..."
107,14,1.168332,"[outlook, demand, ai]",Is it fair to assume that the midpoint of your...,"At the lower end, I will repeat, we have assum..."
45,7,1.167812,"[deal, client]",We won a large deal in this vertical in Q1 to ...,"This time, a lot of our peers have called out ..."
52,7,1.146814,"[outlook, ai]",EURS vertical outlook remains mixed due to eco...,"Financial Services, EURS and Manufacturing gre..."
118,6,1.145198,"[pipeline, cost, client]",While pipeline is strong with new opportunitie...,Pipeline remains healthy with focus on cost ta...
177,21,1.139059,"[outlook, client, ai, uncertainty]","Apurva Prasad So Salil, I am asking if the imp...","Do you see, there is still some scope left if ..."
130,7,1.129128,"[capex, client, ai]","In Services, clients remain cautious about spe...","In Hi-Tech, most clients remain cautious due t..."
58,19,1.123504,"[guidance, client]","Now, in your conversation with the clients, wh...","Salil, are there any silver linings to the cur..."



# 13. FinBERT Tone Analysis

FinBERT is a financial-language model that classifies text as:

- positive
- negative
- neutral

For the MVP, we compare the **average tone of the current call with the previous call**.

Important: this is not the final "tone shift" algorithm. The production version should calculate tone **by topic and section**, not only across the entire transcript.


In [20]:

# ============================================================
# 13. LOAD FINBERT
# ============================================================

FINBERT_NAME = "ProsusAI/finbert"

finbert_tokenizer = AutoTokenizer.from_pretrained(FINBERT_NAME)
finbert_model = AutoModelForSequenceClassification.from_pretrained(
    FINBERT_NAME
).to(DEVICE)

finbert_model.eval()

print("FinBERT loaded on:", DEVICE)


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT loaded on: cuda


In [21]:

# ============================================================
# 14. FINBERT SENTIMENT FUNCTION
# ============================================================

FINBERT_LABELS = {
    0: "positive",
    1: "negative",
    2: "neutral"
}

def finbert_sentiment(texts, batch_size=32):

    results = []

    for start in range(0, len(texts), batch_size):

        batch = texts[start:start + batch_size]

        inputs = finbert_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(DEVICE)
            for k, v in inputs.items()
        }

        with torch.no_grad():
            outputs = finbert_model(**inputs)

        probs = torch.softmax(outputs.logits, dim=-1)

        for p in probs.cpu().numpy():
            label_idx = int(np.argmax(p))

            results.append({
                "label": FINBERT_LABELS[label_idx],
                "positive": float(p[0]),
                "negative": float(p[1]),
                "neutral": float(p[2])
            })

    return pd.DataFrame(results)


# Analyze only sentences with meaningful length.
for filename, df in sentence_tables.items():

    sentiments = finbert_sentiment(
        df["sentence"].tolist()
    )

    df = pd.concat(
        [df.reset_index(drop=True), sentiments],
        axis=1
    )

    sentence_tables[filename] = df

    print(filename)
    print(df["label"].value_counts())
    print()


INFY_Q1FY26.pdf
label
neutral     298
positive    132
negative     40
Name: count, dtype: int64

INFY_Q3FY25.pdf
label
neutral     277
positive    160
negative     39
Name: count, dtype: int64

INFY_Q4FY25.pdf
label
neutral     283
positive    109
negative     55
Name: count, dtype: int64



In [22]:

# ============================================================
# 15. CALL-LEVEL TONE COMPARISON
# ============================================================

def tone_summary(filename):
    df = sentence_tables[filename]

    return {
        "file": filename,
        "positive": df["positive"].mean(),
        "negative": df["negative"].mean(),
        "neutral": df["neutral"].mean()
    }


tone_df = pd.DataFrame([
    tone_summary(filename)
    for filename in ordered_files
])

display(tone_df)

latest_tone = tone_df.iloc[-1]
previous_tone = tone_df.iloc[-2]

print("\nTONE CHANGE")
print("Positive:", round(latest_tone["positive"] - previous_tone["positive"], 4))
print("Negative:", round(latest_tone["negative"] - previous_tone["negative"], 4))
print("Neutral :", round(latest_tone["neutral"] - previous_tone["neutral"], 4))


,file,positive,negative,neutral
0,INFY_Q3FY25.pdf,0.345710,0.104472,0.549818
1,INFY_Q4FY25.pdf,0.276156,0.144992,0.578852
2,INFY_Q1FY26.pdf,0.298868,0.118155,0.582976



TONE CHANGE
Positive: 0.0227
Negative: -0.0268
Neutral : 0.0041



# 16. Topic / Theme Delta — Lightweight MVP

Instead of running a large topic-modeling pipeline immediately, we identify the main finance themes using keyword groups.

This gives us a stable MVP. Later we can replace it with BERTopic / clustering.


In [23]:

# ============================================================
# 16. TOPIC DETECTION
# ============================================================

TOPICS = {
    "Revenue Growth": [
        "revenue", "growth", "constant currency"
    ],
    "Margins": [
        "margin", "ebit", "operating margin", "profitability"
    ],
    "Demand": [
        "demand", "environment", "recovery", "spending"
    ],
    "Deals": [
        "deal", "large deal", "order", "bookings", "contract"
    ],
    "AI / GenAI": [
        "genai", "generative ai", "artificial intelligence", "ai"
    ],
    "Attrition / Hiring": [
        "attrition", "hiring", "headcount", "employees"
    ],
    "Pricing": [
        "pricing", "price", "commercial"
    ],
    "Capex": [
        "capex", "capital expenditure", "investment"
    ],
    "Clients": [
        "client", "customer", "enterprise"
    ],
    "Cost": [
        "cost", "wage", "subcontracting", "efficiency"
    ],
}

def detect_topics(sentence):
    s = sentence.lower()
    found = []

    for topic, keywords in TOPICS.items():
        if any(k in s for k in keywords):
            found.append(topic)

    return found


topic_results = {}

for filename, df in sentence_tables.items():

    topic_counts = {}

    for sentence in df["sentence"]:
        for topic in detect_topics(sentence):
            topic_counts[topic] = topic_counts.get(topic, 0) + 1

    topic_results[filename] = topic_counts

topic_df = pd.DataFrame(topic_results).fillna(0).T

display(topic_df)


,AI / GenAI,Revenue Growth,Deals,Margins,Clients,Demand,Cost,Pricing,Capex,Attrition / Hiring
INFY_Q1FY26.pdf,112,46,45,18,64,35,28,14,11,9
INFY_Q3FY25.pdf,89,52,60,16,62,15,30,12,4,10
INFY_Q4FY25.pdf,105,44,51,38,48,31,38,9,2,6


In [24]:

# ============================================================
# 17. NEW / DROPPED TOPICS
# ============================================================

old_topics = {
    topic for topic, count in topic_results[previous_file].items()
    if count >= 2
}

new_topics = {
    topic for topic, count in topic_results[latest_file].items()
    if count >= 2
}

appeared = sorted(new_topics - old_topics)
disappeared = sorted(old_topics - new_topics)

print("NEW / RE-EMERGING TOPICS")
for topic in appeared:
    print(" +", topic)

print("\nDROPPED / DE-EMPHASIZED TOPICS")
for topic in disappeared:
    print(" -", topic)


NEW / RE-EMERGING TOPICS

DROPPED / DE-EMPHASIZED TOPICS



# 18. Preliminary Guidance Detection

This MVP uses pattern matching to locate guidance-like statements.

It is intentionally conservative.

A production system should extract:

- metric
- value
- period
- direction
- previous guidance
- current guidance
- source page
- confidence

For now, we surface candidate statements for analyst verification.


In [25]:

# ============================================================
# 18. GUIDANCE CANDIDATES
# ============================================================

GUIDANCE_PATTERNS = [
    r"guidance",
    r"expect",
    r"expects",
    r"outlook",
    r"forecast",
    r"we see",
    r"we anticipate",
    r"target",
    r"margin.*range",
    r"revenue.*range",
    r"growth.*range"
]

guidance_regex = re.compile(
    "|".join(GUIDANCE_PATTERNS),
    flags=re.I
)

guidance_candidates = sentence_tables[latest_file][
    sentence_tables[latest_file]["sentence"].apply(
        lambda x: bool(guidance_regex.search(x))
    )
].copy()

display(
    guidance_candidates[
        ["page", "sentence", "label"]
    ].head(30)
)


,page,sentence,label
10,3,Please note that anything we say that refers t...,neutral
34,4,Based on our performance in Q1 and our current...,neutral
35,4,"From the earlier guidance of 0% to 3%, now it ...",positive
36,4,Our margin guidance remains unchanged at 20% t...,neutral
56,5,We expect FY26 free cash flows to be above 100...,positive
71,6,We expect the effective ETR rate for the finan...,neutral
74,6,"Near-term visibility remains good, and we expe...",positive
86,7,EURS vertical outlook remains mixed due to eco...,negative
96,7,Enhanced interest in AI is resulting in budget...,positive
105,7,Driven by our Q1 performance and our current a...,positive



# 19. Analyst-Ready "What Changed?" Brief

This is the MVP's final output.

It does NOT claim that every flagged sentence is important.

Instead, it gives the analyst a ranked shortlist that can be verified against the source transcript.


In [26]:

# ============================================================
# 19. BUILD ANALYST BRIEF
# ============================================================

def make_brief():

    top = important_changes.head(5)

    print("=" * 80)
    print("INFOSYS — EARNINGS CALL DELTA")
    print("=" * 80)

    print(f"Previous: {previous_file}")
    print(f"Latest  : {latest_file}")

    print("\nTONE DELTA")
    print(
        f"Positive: {latest_tone['positive'] - previous_tone['positive']:+.4f}"
    )
    print(
        f"Negative: {latest_tone['negative'] - previous_tone['negative']:+.4f}"
    )

    print("\nTOPIC CHANGES")

    if appeared:
        print("New / re-emerging:", ", ".join(appeared))

    if disappeared:
        print("Dropped / de-emphasized:", ", ".join(disappeared))

    print("\nTOP 5 POTENTIAL MATERIAL CHANGES")
    print("-" * 80)

    for i, (_, row) in enumerate(top.iterrows(), start=1):

        print(f"\n{i}. PAGE {int(row['page'])}")
        print("NEW / CURRENT:")
        print(row["sentence"])

        print("\nPREVIOUS:")
        print(row["matched_old_sentence"])

        print(
            f"\nSemantic similarity: {row['similarity']:.3f}"
        )

        print(
            "Finance keywords:",
            ", ".join(row["keyword_hits"])
        )

        print("-" * 80)


make_brief()


INFOSYS — EARNINGS CALL DELTA
Previous: INFY_Q4FY25.pdf
Latest  : INFY_Q1FY26.pdf

TONE DELTA
Positive: +0.0227
Negative: -0.0268

TOPIC CHANGES

TOP 5 POTENTIAL MATERIAL CHANGES
--------------------------------------------------------------------------------

1. PAGE 6
NEW / CURRENT:
We are partnering with GCCs, both in setup and growth-led deals.

PREVIOUS:
Are you having any conversations with clients around the supply chain solutions or anything like that?

Semantic similarity: 0.419
Finance keywords: growth, deal
--------------------------------------------------------------------------------

2. PAGE 22
NEW / CURRENT:
The hiring for the S&M mainly to improve our growth trajectory, which is what I called out as 20 basis point sales investment in the margin walk.

PREVIOUS:
In terms of your margin walk, you have said the M&A- related cost being 40 bps as a headwind to the margin.

Semantic similarity: 0.533
Finance keywords: growth, margin, hiring, ai
------------------------------

In [27]:

# ============================================================
# 20. SAVE RESULTS
# ============================================================

important_changes.to_csv(
    OUTPUT_DIR / "important_changes.csv",
    index=False
)

tone_df.to_csv(
    OUTPUT_DIR / "tone_comparison.csv",
    index=False
)

topic_df.to_csv(
    OUTPUT_DIR / "topic_comparison.csv"
)

guidance_candidates.to_csv(
    OUTPUT_DIR / "guidance_candidates.csv",
    index=False
)

print("Saved outputs to:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print(" -", p.name)


Saved outputs to:
 - guidance_candidates.csv
 - important_changes.csv
 - tone_comparison.csv
 - topic_comparison.csv



# 21. What this MVP proves

If this notebook works correctly, you now have the first version of the actual research product:

**3 earnings calls → NLP comparison → candidate material changes → tone delta → topic delta → analyst brief.**

### Next upgrades

1. Add 8 quarters instead of 3
2. Build proper management/Q&A segmentation
3. Implement exact numerical guidance extraction
4. Add Loughran-McDonald uncertainty/negative/positive scores
5. Build a Q&A evasion classifier
6. Replace keyword topics with BERTopic
7. Add source-level evidence highlighting
8. Create Streamlit analyst UI
9. Manually label 200–500 changes and calculate precision/recall
10. Compare detected deltas against analyst notes / estimate revisions

**Do not add all of these before validating the core delta detector.**
